# Underlying Price ↔ Option IV Movement EDA

This notebook is designed for your NIFTY IV imputation project. It explores whether **underlying price movement contains predictive signal for option IV movement**, especially for the hard regimes you care about: **Jan 27**, **interior missing cells**, and **edge-like / wing regions**.

The notebook does not assume the signal exists. It builds plots, residual diagnostics, lag studies, bucketed analyses, and validation-style tests so you can decide whether an underlying-price feature is worth adding to the imputer.

Main questions:

1. Do option IVs move systematically after spot moves?
2. Is the effect stronger on Jan 27 / expiry day?
3. Is the relationship different for CE vs PE?
4. Does the signal depend on strike rank / moneyness / local smile slope?
5. Does a simple spot-based correction reduce CV error compared to the current cross-section baseline?


In [ ]:
# ================================================================
# 0. Imports and notebook styling
# ================================================================
import os, re, glob, warnings, math
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

try:
    from scipy.interpolate import PchipInterpolator
    from scipy.stats import spearmanr, pearsonr
    HAS_SCIPY = True
except Exception:
    PchipInterpolator = None
    HAS_SCIPY = False

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

if HAS_SEABORN:
    sns.set_theme(style='whitegrid', context='notebook')

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

print('Ready.')


## 1. Load dataset

Put `dataset.csv` in the project directory, or set `DATA_PATH` manually.

The code below tries relative locations automatically.


In [ ]:
# ================================================================
# 1. Load dataset
# ================================================================

# Change this manually if needed.
DATA_PATH = None

candidate_paths = []
if DATA_PATH is not None:
    candidate_paths.append(DATA_PATH)

candidate_paths += [
    'dataset.csv',
    './dataset.csv',
]

candidate_paths += glob.glob('**/dataset.csv', recursive=True)

DATA_PATH_FOUND = None
for p in candidate_paths:
    if p and Path(p).exists():
        DATA_PATH_FOUND = p
        break

if DATA_PATH_FOUND is None:
    raise FileNotFoundError('Could not find dataset.csv. Set DATA_PATH manually in this cell.')

print('Using:', DATA_PATH_FOUND)
df_raw = pd.read_csv(DATA_PATH_FOUND)
print(df_raw.shape)
df_raw.head()


In [ ]:
# ================================================================
# 2. Parse columns and metadata
# ================================================================

df = df_raw.copy()

if 'datetime' not in df.columns:
    raise ValueError('Expected a datetime column.')
if 'underlying_price' not in df.columns:
    raise ValueError('Expected an underlying_price column.')

df['datetime_parsed'] = pd.to_datetime(df['datetime'], format='%d-%m-%Y %H:%M', errors='coerce')
if df['datetime_parsed'].isna().any():
    print('Warning: exact format failed for some rows. Trying flexible parser.')
    df['datetime_parsed'] = pd.to_datetime(df['datetime'], errors='coerce')

if df['datetime_parsed'].isna().any():
    bad = df[df['datetime_parsed'].isna()].head()
    raise ValueError(f'Unparseable datetimes: {len(bad)} examples shown below:\n{bad}')

df = df.sort_values('datetime_parsed').reset_index(drop=True)

pattern = re.compile(
    r'^(?P<underlying>[A-Z]+)'
    r'(?P<expiry>\d{2}[A-Z]{3}\d{2})'
    r'(?P<strike>\d+)'
    r'(?P<option_type>CE|PE)$'
)

records = []
for col in df.columns:
    if col in {'datetime', 'datetime_parsed', 'underlying_price'}:
        continue
    m = pattern.match(col)
    if m:
        d = m.groupdict()
        d['column'] = col
        d['strike'] = int(d['strike'])
        d['expiry_date'] = pd.to_datetime(d['expiry'], format='%d%b%y', errors='coerce')
        records.append(d)

meta = pd.DataFrame(records).sort_values(['option_type','strike','column']).reset_index(drop=True)
if meta.empty:
    raise ValueError('Could not parse option columns. Check column names.')

option_cols = meta['column'].tolist()
strike_map = dict(zip(meta['column'], meta['strike']))
type_map = dict(zip(meta['column'], meta['option_type']))
cols_by_type = {
    'CE': [c for c in option_cols if type_map[c] == 'CE'],
    'PE': [c for c in option_cols if type_map[c] == 'PE'],
}
cols_by_type['CE'] = sorted(cols_by_type['CE'], key=lambda c: strike_map[c])
cols_by_type['PE'] = sorted(cols_by_type['PE'], key=lambda c: strike_map[c])

print('Rows:', len(df))
print('Option columns:', len(option_cols))
print('CE:', len(cols_by_type['CE']), 'PE:', len(cols_by_type['PE']))
print('Date range:', df['datetime_parsed'].min(), 'to', df['datetime_parsed'].max())
print('Missing IV cells:', int(df[option_cols].isna().sum().sum()))
meta.head()


## 2. Basic market and missingness overview

Before checking signal, we need to understand the dataset structure: dates, intraday timestamps, spot movement, missingness patterns, and whether Jan 27 is actually present after parsing.


In [ ]:
# ================================================================
# 3. Basic derived fields
# ================================================================

df['date'] = df['datetime_parsed'].dt.date
df['time'] = df['datetime_parsed'].dt.time
df['minute_of_day'] = df['datetime_parsed'].dt.hour * 60 + df['datetime_parsed'].dt.minute
df['hour'] = df['datetime_parsed'].dt.hour
df['is_jan27'] = df['datetime_parsed'].dt.date == pd.Timestamp('2026-01-27').date()

S = df['underlying_price'].astype(float)
df['spot_ret_1'] = S.pct_change()
df['spot_logret_1'] = np.log(S).diff()
df['spot_ret_1_bp'] = 10000 * df['spot_ret_1']
df['abs_spot_ret_1_bp'] = df['spot_ret_1_bp'].abs()
df['spot_ret_2'] = S.pct_change(2)
df['spot_ret_5'] = S.pct_change(5)
df['spot_realized_abs_5_bp'] = df['spot_ret_1_bp'].abs().rolling(5, min_periods=1).mean()
df['spot_realized_abs_15_bp'] = df['spot_ret_1_bp'].abs().rolling(15, min_periods=1).mean()

summary = pd.DataFrame({
    'rows': [len(df)],
    'dates': [df['date'].nunique()],
    'jan27_rows': [int(df['is_jan27'].sum())],
    'missing_cells': [int(df[option_cols].isna().sum().sum())],
    'observed_cells': [int(df[option_cols].notna().sum().sum())],
    'spot_min': [S.min()],
    'spot_max': [S.max()],
    'spot_total_return_pct': [(S.iloc[-1]/S.iloc[0]-1)*100],
})
summary


In [ ]:
# ================================================================
# 4. Plot spot path and returns
# ================================================================

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['datetime_parsed'], df['underlying_price'], linewidth=1.5)
ax.set_title('Underlying Price Path')
ax.set_xlabel('Timestamp')
ax.set_ylabel('Underlying price')
plt.xticks(rotation=30)
plt.show()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['datetime_parsed'], df['spot_ret_1_bp'], linewidth=1)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('One-step Spot Return in Basis Points')
ax.set_xlabel('Timestamp')
ax.set_ylabel('Return, bp')
plt.xticks(rotation=30)
plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(df['spot_ret_1_bp'].dropna(), bins=80)
ax.set_title('Distribution of One-step Spot Returns')
ax.set_xlabel('Return, bp')
ax.set_ylabel('Count')
plt.show()


In [ ]:
# Missingness by time and option type
miss_records = []
for i, row in df.iterrows():
    for ot in ['CE','PE']:
        cols = cols_by_type[ot]
        miss_records.append({
            'row_idx': i,
            'datetime': row['datetime_parsed'],
            'date': row['date'],
            'is_jan27': row['is_jan27'],
            'option_type': ot,
            'missing_count': int(row[cols].isna().sum()),
            'observed_count': int(row[cols].notna().sum()),
        })
miss_df = pd.DataFrame(miss_records)

fig, ax = plt.subplots(figsize=(14,4))
for ot, sub in miss_df.groupby('option_type'):
    ax.plot(sub['datetime'], sub['missing_count'], label=ot, linewidth=1)
ax.set_title('Missing Option IV Count per Timestamp')
ax.set_ylabel('Missing count')
ax.legend()
plt.xticks(rotation=30)
plt.show()

pivot = miss_df.pivot_table(index='date', columns='option_type', values='missing_count', aggfunc='mean')
display(pivot)


## 3. Convert the option panel into a long table

The long table is the core EDA object. Every row is one contract at one timestamp.

Features created:

- option type: CE / PE
- strike
- strike rank within CE/PE
- moneyness = strike / underlying price
- IV level
- IV change over time for the same contract
- spot return over time
- Jan 27 flag
- local smile slope from neighboring strikes


In [ ]:
# ================================================================
# 5. Build long table with option IV movements and spot movements
# ================================================================

long_parts = []
for ot in ['CE', 'PE']:
    cols = cols_by_type[ot]
    tmp = df[['datetime', 'datetime_parsed', 'date', 'time', 'minute_of_day', 'hour', 'is_jan27',
              'underlying_price', 'spot_ret_1', 'spot_logret_1', 'spot_ret_1_bp', 'abs_spot_ret_1_bp',
              'spot_ret_2', 'spot_ret_5', 'spot_realized_abs_5_bp', 'spot_realized_abs_15_bp'] + cols].copy()
    melted = tmp.melt(
        id_vars=['datetime', 'datetime_parsed', 'date', 'time', 'minute_of_day', 'hour', 'is_jan27',
                 'underlying_price', 'spot_ret_1', 'spot_logret_1', 'spot_ret_1_bp', 'abs_spot_ret_1_bp',
                 'spot_ret_2', 'spot_ret_5', 'spot_realized_abs_5_bp', 'spot_realized_abs_15_bp'],
        value_vars=cols,
        var_name='contract',
        value_name='iv'
    )
    melted['option_type'] = ot
    long_parts.append(melted)

long = pd.concat(long_parts, ignore_index=True)
long['strike'] = long['contract'].map(strike_map).astype(int)
long['moneyness'] = long['strike'] / long['underlying_price']
long['log_moneyness'] = np.log(long['moneyness'])

# rank within option type
rank_map = {}
for ot in ['CE','PE']:
    for k, c in enumerate(cols_by_type[ot]):
        rank_map[c] = k
long['k_rank'] = long['contract'].map(rank_map).astype(int)
long['n_rank'] = long['option_type'].map(lambda ot: len(cols_by_type[ot]))
long['rank_frac'] = long['k_rank'] / (long['n_rank'] - 1)
long['is_left_rank'] = long['k_rank'] == 0
long['is_right_rank'] = long['k_rank'] == (long['n_rank'] - 1)
long['is_wing_rank'] = long['is_left_rank'] | long['is_right_rank']

# IV time movement by same contract
long = long.sort_values(['contract','datetime_parsed']).reset_index(drop=True)
long['iv_lag1'] = long.groupby('contract')['iv'].shift(1)
long['iv_lag2'] = long.groupby('contract')['iv'].shift(2)
long['iv_change_1'] = long['iv'] - long['iv_lag1']
long['iv_pct_change_1'] = long.groupby('contract')['iv'].pct_change()
long['abs_iv_change_1'] = long['iv_change_1'].abs()

# Lagged spot features aligned with current IV movement
long['spot_ret_1_lag0_bp'] = long['spot_ret_1_bp']
long['spot_ret_1_lag1_bp'] = long.groupby('contract')['spot_ret_1_bp'].shift(1)
long['spot_ret_1_lag2_bp'] = long.groupby('contract')['spot_ret_1_bp'].shift(2)
long['spot_ret_1_lead1_bp'] = long.groupby('contract')['spot_ret_1_bp'].shift(-1)

long_obs = long[np.isfinite(long['iv'])].copy()
long_move = long_obs[np.isfinite(long_obs['iv_change_1']) & np.isfinite(long_obs['spot_ret_1_bp'])].copy()

print('Long table:', long.shape)
print('Observed long:', long_obs.shape)
print('Movement rows:', long_move.shape)
long_obs.head()


In [ ]:
# ================================================================
# 6. Add local smile slope per timestamp / option type
# ================================================================

def add_local_smile_slopes(long_df):
    out = []
    for (dt, ot), g in long_df.groupby(['datetime_parsed','option_type'], sort=False):
        gg = g.sort_values('moneyness').copy()
        x = gg['moneyness'].to_numpy(float)
        y = gg['iv'].to_numpy(float)
        slope = np.full(len(gg), np.nan)
        curvature = np.full(len(gg), np.nan)
        ok = np.isfinite(x) & np.isfinite(y)
        idxs = np.where(ok)[0]
        if len(idxs) >= 2:
            # nearest finite neighbors around each point; central finite difference where possible
            for pos in range(len(gg)):
                if not np.isfinite(x[pos]):
                    continue
                left = idxs[idxs < pos]
                right = idxs[idxs > pos]
                if len(left) and len(right):
                    l, r = left[-1], right[0]
                    if x[r] != x[l]:
                        slope[pos] = (y[r] - y[l]) / (x[r] - x[l])
                elif len(left):
                    l = left[-1]
                    if x[pos] != x[l] and np.isfinite(y[pos]):
                        slope[pos] = (y[pos] - y[l]) / (x[pos] - x[l])
                elif len(right):
                    r = right[0]
                    if x[r] != x[pos] and np.isfinite(y[pos]):
                        slope[pos] = (y[r] - y[pos]) / (x[r] - x[pos])
            # rough second derivative using gradient over observed points
            if len(idxs) >= 3:
                try:
                    dy = np.gradient(y[idxs], x[idxs])
                    d2 = np.gradient(dy, x[idxs])
                    curvature[idxs] = d2
                except Exception:
                    pass
        gg['local_smile_slope'] = slope
        gg['local_smile_curvature'] = curvature
        out.append(gg)
    return pd.concat(out, ignore_index=True)

long2 = add_local_smile_slopes(long)
long_obs2 = long2[np.isfinite(long2['iv'])].copy()
long_move2 = long_obs2[np.isfinite(long_obs2['iv_change_1']) & np.isfinite(long_obs2['spot_ret_1_bp'])].copy()

# Moneyness drift from previous timestamp for same strike due to spot movement
long_move2['moneyness_lag1'] = long_move2.groupby('contract')['moneyness'].shift(1)
long_move2['dmoneyness_1'] = long_move2['moneyness'] - long_move2['moneyness_lag1']
long_move2['slope_x_dmoneyness'] = long_move2['local_smile_slope'] * long_move2['dmoneyness_1']
long_move2['abs_slope_x_dmoneyness'] = long_move2['slope_x_dmoneyness'].abs()

long_move2[['datetime_parsed','contract','option_type','strike','k_rank','iv','iv_change_1','spot_ret_1_bp','moneyness','local_smile_slope','slope_x_dmoneyness']].head()


## 4. First-order relationship: spot return vs IV change

This section checks the simplest possible relationship:

\[
\Delta IV_{t,c} = IV_{t,c} - IV_{t-1,c}
\]

against

\[
\Delta S_t / S_{t-1}
\]

We inspect it globally, by option type, by date regime, and by strike rank.


In [ ]:
# ================================================================
# 7. Correlation summary helpers
# ================================================================

def corr_safe(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 5:
        return pd.Series({'n': int(m.sum()), 'pearson': np.nan, 'spearman': np.nan})
    xx = np.asarray(x)[m]
    yy = np.asarray(y)[m]
    try:
        pear = pearsonr(xx, yy)[0] if HAS_SCIPY else np.corrcoef(xx, yy)[0,1]
    except Exception:
        pear = np.nan
    try:
        spear = spearmanr(xx, yy)[0] if HAS_SCIPY else pd.Series(xx).corr(pd.Series(yy), method='spearman')
    except Exception:
        spear = np.nan
    return pd.Series({'n': int(m.sum()), 'pearson': pear, 'spearman': spear})

features_to_test = [
    'spot_ret_1_bp', 'abs_spot_ret_1_bp', 'spot_realized_abs_5_bp', 'spot_realized_abs_15_bp',
    'spot_ret_1_lag1_bp', 'spot_ret_1_lag2_bp', 'slope_x_dmoneyness', 'abs_slope_x_dmoneyness',
]

rows = []
for feat in features_to_test:
    res = corr_safe(long_move2[feat], long_move2['iv_change_1'])
    rows.append({'feature': feat, 'target': 'iv_change_1', **res.to_dict()})

corr_global = pd.DataFrame(rows).sort_values('pearson', key=lambda s: s.abs(), ascending=False)
corr_global


In [ ]:
# By option type and Jan27 regime
rows = []
for keys, g in long_move2.groupby(['option_type','is_jan27']):
    ot, isj = keys
    for feat in features_to_test:
        res = corr_safe(g[feat], g['iv_change_1'])
        rows.append({'option_type': ot, 'is_jan27': isj, 'feature': feat, **res.to_dict()})

corr_by_regime = pd.DataFrame(rows)
corr_by_regime.sort_values(['is_jan27','option_type','pearson'], key=lambda s: s.abs() if s.name=='pearson' else s, ascending=False).head(40)


In [ ]:
# Heatmap: correlation of spot return with IV change by option type and strike rank
heat = long_move2.groupby(['option_type','k_rank']).apply(
    lambda g: corr_safe(g['spot_ret_1_bp'], g['iv_change_1'])['pearson']
).reset_index(name='corr')

pivot = heat.pivot(index='option_type', columns='k_rank', values='corr')
fig, ax = plt.subplots(figsize=(14, 3.2))
if HAS_SEABORN:
    sns.heatmap(pivot, annot=True, fmt='.2f', center=0, cmap='coolwarm', ax=ax)
else:
    im = ax.imshow(pivot.values, aspect='auto')
    ax.set_xticks(range(pivot.shape[1])); ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(pivot.shape[0])); ax.set_yticklabels(pivot.index)
    fig.colorbar(im, ax=ax)
ax.set_title('Pearson Corr: Spot Return bp vs Same-contract IV Change, by Strike Rank')
plt.show()

# Same but Jan27 only
heat_j = long_move2[long_move2['is_jan27']].groupby(['option_type','k_rank']).apply(
    lambda g: corr_safe(g['spot_ret_1_bp'], g['iv_change_1'])['pearson']
).reset_index(name='corr')
if len(heat_j):
    pivot_j = heat_j.pivot(index='option_type', columns='k_rank', values='corr')
    fig, ax = plt.subplots(figsize=(14, 3.2))
    if HAS_SEABORN:
        sns.heatmap(pivot_j, annot=True, fmt='.2f', center=0, cmap='coolwarm', ax=ax)
    else:
        im = ax.imshow(pivot_j.values, aspect='auto')
        ax.set_xticks(range(pivot_j.shape[1])); ax.set_xticklabels(pivot_j.columns)
        ax.set_yticks(range(pivot_j.shape[0])); ax.set_yticklabels(pivot_j.index)
        fig.colorbar(im, ax=ax)
    ax.set_title('Jan27 Only: Corr Spot Return bp vs IV Change')
    plt.show()
else:
    print('No Jan27 rows detected.')


In [ ]:
# Scatter plots with binned mean overlay

def scatter_with_bins(data, x, y, title, bins=20, sample=8000):
    d = data[[x,y,'option_type','is_jan27']].dropna().copy()
    if len(d) > sample:
        d_plot = d.sample(sample, random_state=42)
    else:
        d_plot = d
    fig, ax = plt.subplots(figsize=(12,5))
    for key, sub in d_plot.groupby('option_type'):
        ax.scatter(sub[x], sub[y], s=9, alpha=0.25, label=key)
    # bin means
    try:
        d['bin'] = pd.qcut(d[x], bins, duplicates='drop')
        bm = d.groupby('bin').agg(x_mean=(x,'mean'), y_mean=(y,'mean'), n=(y,'size')).reset_index()
        ax.plot(bm['x_mean'], bm['y_mean'], linewidth=2.5, marker='o', label='binned mean')
    except Exception as e:
        pass
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.legend()
    plt.show()

scatter_with_bins(long_move2, 'spot_ret_1_bp', 'iv_change_1', 'Spot return vs IV change')
scatter_with_bins(long_move2[long_move2['is_jan27']], 'spot_ret_1_bp', 'iv_change_1', 'Jan27: Spot return vs IV change')
scatter_with_bins(long_move2, 'slope_x_dmoneyness', 'iv_change_1', 'Smile-slope × moneyness drift vs IV change')


## 5. Directional asymmetry: CE vs PE, spot up vs spot down

A useful signal may be asymmetric. For example:

- when spot goes up, calls may behave differently than puts,
- expiry-day behavior may differ from normal days,
- far OTM wings may react differently from central strikes.


In [ ]:
# ================================================================
# 8. Bucketed IV movement by spot return sign and magnitude
# ================================================================

d = long_move2.copy()
d['spot_direction'] = np.where(d['spot_ret_1_bp'] > 0, 'spot_up', np.where(d['spot_ret_1_bp'] < 0, 'spot_down', 'flat'))
d['spot_move_bucket'] = pd.cut(
    d['spot_ret_1_bp'],
    bins=[-np.inf, -20, -10, -5, 0, 5, 10, 20, np.inf],
    labels=['<-20','-20..-10','-10..-5','-5..0','0..5','5..10','10..20','>20']
)

bucket_summary = d.groupby(['is_jan27','option_type','spot_move_bucket']).agg(
    n=('iv_change_1','size'),
    mean_iv_change=('iv_change_1','mean'),
    median_iv_change=('iv_change_1','median'),
    mean_abs_iv_change=('abs_iv_change_1','mean'),
    mean_spot_bp=('spot_ret_1_bp','mean'),
).reset_index()

bucket_summary.head(20)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, isj in zip(axes, [False, True]):
    sub = bucket_summary[bucket_summary['is_jan27'] == isj]
    if sub.empty:
        ax.text(0.5, 0.5, f'No data for is_jan27={isj}', ha='center')
        continue
    for ot in ['CE','PE']:
        ss = sub[sub['option_type'] == ot]
        ax.plot(ss['spot_move_bucket'].astype(str), ss['mean_iv_change'], marker='o', label=ot)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'Mean IV Change by Spot Return Bucket | Jan27={isj}')
    ax.set_ylabel('Mean ΔIV')
    ax.legend()
plt.xticks(rotation=30)
plt.show()


In [ ]:
# Strike-rank view: mean IV change after spot up/down
rank_summary = d[d['spot_direction'].isin(['spot_up','spot_down'])].groupby(
    ['is_jan27','option_type','spot_direction','k_rank']
).agg(
    n=('iv_change_1','size'),
    mean_iv_change=('iv_change_1','mean'),
    median_iv_change=('iv_change_1','median'),
    mean_abs_iv_change=('abs_iv_change_1','mean'),
).reset_index()

for isj in [False, True]:
    sub = rank_summary[rank_summary['is_jan27'] == isj]
    if sub.empty: 
        continue
    fig, axes = plt.subplots(1, 2, figsize=(15,4), sharey=True)
    for ax, ot in zip(axes, ['CE','PE']):
        ss = sub[sub['option_type'] == ot]
        for direction in ['spot_up','spot_down']:
            ssd = ss[ss['spot_direction'] == direction]
            ax.plot(ssd['k_rank'], ssd['mean_iv_change'], marker='o', label=direction)
        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_title(f'{ot}: mean ΔIV by rank | Jan27={isj}')
        ax.set_xlabel('k_rank')
        ax.set_ylabel('Mean ΔIV')
        ax.legend()
    plt.show()


## 6. Surface-level movement: IV level, skew, curvature, and spot

Instead of looking at each contract separately, this section summarizes each timestamp's smile using:

- mean IV
- median IV
- wing IVs
- CE/PE skew proxy
- local slopes and curvatures

Then we compare those surface features to underlying movement.


In [ ]:
# ================================================================
# 9. Timestamp-level surface features
# ================================================================

def row_surface_features(row, ot):
    cols = cols_by_type[ot]
    vals = row[cols].astype(float)
    obs = vals.dropna()
    out = {}
    out[f'{ot}_obs_n'] = int(obs.shape[0])
    out[f'{ot}_mean_iv'] = float(obs.mean()) if len(obs) else np.nan
    out[f'{ot}_median_iv'] = float(obs.median()) if len(obs) else np.nan
    out[f'{ot}_std_iv'] = float(obs.std()) if len(obs) > 1 else np.nan
    # ranks
    if len(cols):
        out[f'{ot}_left_iv'] = row[cols[0]]
        out[f'{ot}_right_iv'] = row[cols[-1]]
        out[f'{ot}_center_iv'] = row[cols[len(cols)//2]]
        # simple skew / wing proxies
        if pd.notna(row[cols[0]]) and pd.notna(row[cols[-1]]):
            out[f'{ot}_right_minus_left'] = float(row[cols[-1]] - row[cols[0]])
        else:
            out[f'{ot}_right_minus_left'] = np.nan
        if pd.notna(row[cols[0]]) and pd.notna(row[cols[len(cols)//2]]) and pd.notna(row[cols[-1]]):
            out[f'{ot}_wing_minus_center'] = float(0.5*(row[cols[0]] + row[cols[-1]]) - row[cols[len(cols)//2]])
        else:
            out[f'{ot}_wing_minus_center'] = np.nan
    return out

surf_records = []
for i, row in df.iterrows():
    rec = {
        'row_idx': i,
        'datetime': row['datetime_parsed'],
        'date': row['date'],
        'minute_of_day': row['minute_of_day'],
        'is_jan27': row['is_jan27'],
        'underlying_price': row['underlying_price'],
        'spot_ret_1_bp': row['spot_ret_1_bp'],
        'abs_spot_ret_1_bp': row['abs_spot_ret_1_bp'],
        'spot_realized_abs_5_bp': row['spot_realized_abs_5_bp'],
        'spot_realized_abs_15_bp': row['spot_realized_abs_15_bp'],
    }
    rec.update(row_surface_features(row, 'CE'))
    rec.update(row_surface_features(row, 'PE'))
    surf_records.append(rec)

surf = pd.DataFrame(surf_records).sort_values('datetime').reset_index(drop=True)
for col in [c for c in surf.columns if c.endswith('_iv') or 'minus' in c or c.endswith('_std_iv')]:
    surf[f'd_{col}'] = surf[col].diff()

surf.head()


In [ ]:
# Plot surface features over time
features_plot = ['CE_median_iv','PE_median_iv','CE_right_minus_left','PE_right_minus_left','CE_wing_minus_center','PE_wing_minus_center']
for feat in features_plot:
    if feat not in surf.columns: 
        continue
    fig, ax = plt.subplots(figsize=(14,4))
    ax.plot(surf['datetime'], surf[feat], linewidth=1.3)
    ax.set_title(f'Time Series: {feat}')
    ax.set_xlabel('Timestamp')
    ax.set_ylabel(feat)
    plt.xticks(rotation=30)
    plt.show()


In [ ]:
# Correlation between spot moves and surface-feature changes
surf_targets = [c for c in surf.columns if c.startswith('d_') and surf[c].notna().sum() > 10]
rows = []
for tgt in surf_targets:
    for feat in ['spot_ret_1_bp','abs_spot_ret_1_bp','spot_realized_abs_5_bp','spot_realized_abs_15_bp']:
        res = corr_safe(surf[feat], surf[tgt])
        rows.append({'surface_change': tgt, 'spot_feature': feat, **res.to_dict()})

surf_corr = pd.DataFrame(rows)
surf_corr['abs_pearson'] = surf_corr['pearson'].abs()
surf_corr.sort_values('abs_pearson', ascending=False).head(30)


## 7. Lag study: does spot lead IV or IV lead spot?

For imputation, we care about whether recent spot movement helps predict current IV. This section checks correlations with different lags.

For each contract we compare:

- current spot return vs current IV change,
- previous spot return vs current IV change,
- previous 2 returns,
- rolling absolute spot movement.


In [ ]:
# ================================================================
# 10. Lag study by option type and strike rank
# ================================================================

lag_features = [
    'spot_ret_1_lag0_bp',
    'spot_ret_1_lag1_bp',
    'spot_ret_1_lag2_bp',
    'spot_ret_1_lead1_bp',
    'abs_spot_ret_1_bp',
    'spot_realized_abs_5_bp',
    'spot_realized_abs_15_bp',
    'slope_x_dmoneyness',
]

lag_rows = []
for (ot, k), g in long_move2.groupby(['option_type','k_rank']):
    for feat in lag_features:
        res = corr_safe(g[feat], g['iv_change_1'])
        lag_rows.append({'option_type': ot, 'k_rank': k, 'feature': feat, **res.to_dict()})
lag_corr = pd.DataFrame(lag_rows)

# Display strongest absolute correlations per rank
lag_corr['abs_pearson'] = lag_corr['pearson'].abs()
lag_corr.sort_values('abs_pearson', ascending=False).head(30)


In [ ]:
# Heatmap of best lag feature per rank based on abs corr
for feat in lag_features:
    pvt = lag_corr[lag_corr['feature'] == feat].pivot(index='option_type', columns='k_rank', values='pearson')
    fig, ax = plt.subplots(figsize=(14,3.2))
    if HAS_SEABORN:
        sns.heatmap(pvt, annot=True, fmt='.2f', center=0, cmap='coolwarm', ax=ax)
    else:
        im = ax.imshow(pvt.values, aspect='auto')
        ax.set_xticks(range(pvt.shape[1])); ax.set_xticklabels(pvt.columns)
        ax.set_yticks(range(pvt.shape[0])); ax.set_yticklabels(pvt.index)
        fig.colorbar(im, ax=ax)
    ax.set_title(f'Corr with IV Change: {feat}')
    plt.show()


## 8. Residual-oriented EDA using your current cross-section baseline

This is the most important part for model improvement.

Raw IV movement correlation is interesting, but your imputer already uses the current cross-section. The real question is:

> After the current cross-section model predicts a masked observed IV, does underlying movement explain the residual?

We simulate missing cells by masking observed values, then compute:

\[
residual = true\_iv - cross\_section\_prediction
\]

Then we test whether spot-derived features explain that residual.


In [ ]:
# ================================================================
# 11. Baseline cross-section predictor copied/simplified from current best
#     Interior: WLS + optional PCHIP blend, edge not focus here
# ================================================================

EPS_IV = 1e-6
BANDWIDTH_GRID = np.array([5e-5, 7e-5, 1e-4, 1.5e-4, 2e-4], dtype=float)
LOCAL_POLY_DEGREE = 2
PCHIP_INTERIOR_WEIGHT = 0.25
MIN_PCHIP_POINTS = 4

def safe_iv_local(x):
    if not np.isfinite(x): return np.nan
    return max(float(x), EPS_IV)

def local_poly_wls_pred(x_obs, y_obs, x_target, bandwidth, degree=2):
    x_obs = np.asarray(x_obs, float); y_obs = np.asarray(y_obs, float)
    m = np.isfinite(x_obs) & np.isfinite(y_obs)
    x_obs, y_obs = x_obs[m], y_obs[m]
    if len(y_obs) == 0: return np.nan
    if len(y_obs) == 1: return safe_iv_local(y_obs[0])
    d = min(degree, len(y_obs)-1)
    dx = x_obs - x_target
    w = np.exp(-dx**2/(2*bandwidth))
    X = np.column_stack([dx**j for j in range(d+1)])
    WX = X * w[:, None]
    try:
        coef = np.linalg.solve(X.T @ WX, X.T @ (w*y_obs))
        return safe_iv_local(coef[0])
    except Exception:
        ws = w.sum()
        return safe_iv_local((w @ y_obs) / ws) if ws > 1e-15 else np.nan

def loo_bw(x, y, grid=BANDWIDTH_GRID):
    x = np.asarray(x, float); y = np.asarray(y, float)
    if len(y) <= 2:
        return float(grid[len(grid)//2])
    best_bw, best_mse = float(grid[len(grid)//2]), np.inf
    for bw in grid:
        errs = []
        for i in range(len(y)):
            pred = local_poly_wls_pred(np.delete(x,i), np.delete(y,i), x[i], bw, degree=2)
            if np.isfinite(pred):
                errs.append((pred-y[i])**2)
        if errs:
            mse = float(np.mean(errs))
            if mse < best_mse:
                best_mse, best_bw = mse, float(bw)
    return best_bw

def pchip_pred(x_obs, y_obs, x_target):
    if PchipInterpolator is None:
        return np.nan
    x = np.asarray(x_obs, float); y = np.asarray(y_obs, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if len(y) < MIN_PCHIP_POINTS:
        return np.nan
    order = np.argsort(x)
    x, y = x[order], y[order]
    ux, inv = np.unique(x, return_inverse=True)
    if len(ux) < MIN_PCHIP_POINTS:
        return np.nan
    if len(ux) != len(x):
        yy = np.zeros(len(ux)); cc = np.zeros(len(ux))
        for i, gi in enumerate(inv):
            yy[gi] += y[i]; cc[gi] += 1
        x, y = ux, yy / np.maximum(cc, 1)
    else:
        x = ux
    if not (x[0] <= x_target <= x[-1]):
        return np.nan
    try:
        return safe_iv_local(float(PchipInterpolator(x, y, extrapolate=False)(x_target)))
    except Exception:
        return np.nan

def collect_row_points_from_row(row, ot):
    spot = row['underlying_price']
    if pd.isna(spot) or spot <= 0:
        return np.array([]), np.array([]), []
    cols = cols_by_type[ot]
    obs_cols = [c for c in cols if pd.notna(row[c])]
    x = np.array([strike_map[c] / spot for c in obs_cols], float)
    y = np.array([row[c] for c in obs_cols], float)
    m = np.isfinite(x) & np.isfinite(y)
    return x[m], y[m], [c for c, ok in zip(obs_cols, m) if ok]

def cross_section_pred_for_masked(df_like, r, c, use_pchip=True):
    row = df_like.loc[r]
    ot = type_map[c]
    spot = row['underlying_price']
    global_med = float(df_like[option_cols].stack().median())
    x_obs, y_obs, _ = collect_row_points_from_row(row, ot)
    if pd.isna(spot) or spot <= 0 or len(y_obs) == 0:
        return global_med, np.nan, np.nan, False
    xt = strike_map[c] / spot
    bw = loo_bw(x_obs, y_obs)
    wls = local_poly_wls_pred(x_obs, y_obs, xt, bw, degree=2)
    if not np.isfinite(wls):
        wls = global_med
    pc = pchip_pred(x_obs, y_obs, xt)
    if use_pchip and np.isfinite(pc):
        pred = (1-PCHIP_INTERIOR_WEIGHT)*wls + PCHIP_INTERIOR_WEIGHT*pc
        return safe_iv_local(pred), wls, pc, True
    return safe_iv_local(wls), wls, pc, False

def is_edge_under_mask(row, c, ot):
    cols = cols_by_type[ot]
    vals = [pd.notna(row[x]) for x in cols]
    i = cols.index(c)
    return not any(vals[:i]) or not any(vals[i+1:])

print('Baseline predictor helpers ready.')


In [ ]:
# ================================================================
# 12. Build masked-residual validation dataset
# ================================================================

def build_residual_dataset(mask_frac=0.08, reps=5, seed=42, only_interior=True, max_records=None):
    rng = np.random.default_rng(seed)
    observed = [(r,c) for r in df.index for c in option_cols if pd.notna(df.at[r,c])]
    records = []
    for rep in range(reps):
        k = int(len(observed) * mask_frac)
        chosen = rng.choice(len(observed), k, replace=False)
        mask_set = [observed[i] for i in chosen]
        df_m = df.copy()
        truths = {}
        for r,c in mask_set:
            truths[(r,c)] = float(df.at[r,c])
            df_m.at[r,c] = np.nan
        for r,c in mask_set:
            ot = type_map[c]
            edge = is_edge_under_mask(df_m.loc[r], c, ot)
            if only_interior and edge:
                continue
            pred, wls, pc, pchip_used = cross_section_pred_for_masked(df_m, r, c, use_pchip=True)
            true = truths[(r,c)]
            residual = true - pred
            row = df.loc[r]
            # Local slope from original full-ish row at the target timestamp; for CV this is allowed only as EDA.
            # In modeling, recompute from df_m if using as a predictor.
            tmp = long2[(long2['datetime_parsed'] == row['datetime_parsed']) & (long2['contract'] == c)]
            local_slope = tmp['local_smile_slope'].iloc[0] if len(tmp) else np.nan
            # moneyness drift
            if r >= 1:
                prev_spot = df.at[r-1, 'underlying_price']
                dm = strike_map[c]/row['underlying_price'] - strike_map[c]/prev_spot if prev_spot > 0 else np.nan
            else:
                dm = np.nan
            records.append({
                'rep': rep,
                'row_idx': r,
                'datetime': row['datetime_parsed'],
                'date': row['date'],
                'minute_of_day': row['minute_of_day'],
                'is_jan27': row['is_jan27'],
                'contract': c,
                'option_type': ot,
                'strike': strike_map[c],
                'k_rank': rank_map[c],
                'true_iv': true,
                'cross_pred': pred,
                'wls_pred': wls,
                'pchip_pred': pc,
                'pchip_used': pchip_used,
                'residual': residual,
                'sq_err': residual**2,
                'spot_ret_1_bp': row['spot_ret_1_bp'],
                'abs_spot_ret_1_bp': row['abs_spot_ret_1_bp'],
                'spot_realized_abs_5_bp': row['spot_realized_abs_5_bp'],
                'spot_realized_abs_15_bp': row['spot_realized_abs_15_bp'],
                'underlying_price': row['underlying_price'],
                'moneyness': strike_map[c] / row['underlying_price'],
                'dmoneyness_1': dm,
                'local_smile_slope': local_slope,
                'slope_x_dmoneyness': local_slope * dm if np.isfinite(local_slope) and np.isfinite(dm) else np.nan,
                'edge_under_mask': edge,
            })
            if max_records and len(records) >= max_records:
                return pd.DataFrame(records)
    return pd.DataFrame(records)

resid_df = build_residual_dataset(mask_frac=0.08, reps=5, seed=42, only_interior=True)
print(resid_df.shape)
print('Baseline interior CV MSE:', resid_df['sq_err'].mean())
resid_df.head()


In [ ]:
# Residual correlations: which spot features explain prediction error?
resid_features = [
    'spot_ret_1_bp', 'abs_spot_ret_1_bp', 'spot_realized_abs_5_bp', 'spot_realized_abs_15_bp',
    'dmoneyness_1', 'local_smile_slope', 'slope_x_dmoneyness', 'moneyness', 'k_rank', 'minute_of_day'
]
rows = []
for feat in resid_features:
    res = corr_safe(resid_df[feat], resid_df['residual'])
    rows.append({'feature': feat, **res.to_dict()})
resid_corr = pd.DataFrame(rows)
resid_corr['abs_pearson'] = resid_corr['pearson'].abs()
resid_corr.sort_values('abs_pearson', ascending=False)


In [ ]:
# Residual correlations by option type and Jan27
rows = []
for (ot, isj), g in resid_df.groupby(['option_type','is_jan27']):
    for feat in resid_features:
        res = corr_safe(g[feat], g['residual'])
        rows.append({'option_type': ot, 'is_jan27': isj, 'feature': feat, **res.to_dict()})
resid_corr_regime = pd.DataFrame(rows)
resid_corr_regime['abs_pearson'] = resid_corr_regime['pearson'].abs()
resid_corr_regime.sort_values('abs_pearson', ascending=False).head(40)


In [ ]:
# Residual scatter plots
for feat in ['spot_ret_1_bp', 'abs_spot_ret_1_bp', 'slope_x_dmoneyness', 'spot_realized_abs_5_bp']:
    scatter_with_bins(resid_df, feat, 'residual', f'Residual vs {feat}', bins=20, sample=12000)


## 9. CV-gated correction tests

This section tries simple corrections and only accepts them if they reduce validation MSE.

We test corrections of the form:

\[
\hat{IV}_{new} = \hat{IV}_{cross} + \alpha \cdot signal
\]

or per-bucket/per-option versions.

Important: if the best alpha is zero or MSE gets worse, the signal should **not** be added to the imputer.


In [ ]:
# ================================================================
# 13. Simple alpha grid correction tests
# ================================================================

def eval_alpha_grid(data, signal_col, group_cols=None, alpha_grid=None, min_n=30):
    if alpha_grid is None:
        alpha_grid = np.round(np.arange(-2.0, 2.0001, 0.05), 3)
    d = data.copy()
    d = d[np.isfinite(d['true_iv']) & np.isfinite(d['cross_pred']) & np.isfinite(d[signal_col])].copy()
    if d.empty:
        return pd.DataFrame(), pd.DataFrame()
    if group_cols is None:
        group_cols = []
    if not group_cols:
        d['_group'] = 'all'
        group_cols = ['_group']
    results = []
    pred_records = []
    for key, g in d.groupby(group_cols):
        if len(g) < min_n:
            best_alpha = 0.0
            best_mse = ((g['cross_pred'] - g['true_iv'])**2).mean()
        else:
            base_mse = ((g['cross_pred'] - g['true_iv'])**2).mean()
            best_alpha, best_mse = 0.0, base_mse
            for a in alpha_grid:
                pred = g['cross_pred'] + float(a) * g[signal_col]
                mse = ((pred - g['true_iv'])**2).mean()
                if mse < best_mse:
                    best_mse = float(mse)
                    best_alpha = float(a)
        key_tuple = key if isinstance(key, tuple) else (key,)
        rec = {col: val for col, val in zip(group_cols, key_tuple)}
        rec.update({
            'signal': signal_col,
            'n': len(g),
            'base_mse': ((g['cross_pred'] - g['true_iv'])**2).mean(),
            'best_mse': best_mse,
            'best_alpha': best_alpha,
            'improvement_pct': (((g['cross_pred'] - g['true_iv'])**2).mean() - best_mse) / max(((g['cross_pred'] - g['true_iv'])**2).mean(), 1e-18) * 100,
        })
        results.append(rec)
    summary = pd.DataFrame(results)
    return summary, d

signals = ['spot_ret_1_bp', 'abs_spot_ret_1_bp', 'dmoneyness_1', 'slope_x_dmoneyness', 'spot_realized_abs_5_bp']
all_summaries = []
for sig in signals:
    for group_cols in [None, ['option_type'], ['option_type','k_rank'], ['is_jan27','option_type'], ['is_jan27','option_type','k_rank']]:
        summ, _ = eval_alpha_grid(resid_df, sig, group_cols=group_cols, min_n=25)
        if len(summ):
            summ['grouping'] = 'global' if group_cols is None else '+'.join(group_cols)
            all_summaries.append(summ)

alpha_summary = pd.concat(all_summaries, ignore_index=True)
alpha_summary.sort_values('improvement_pct', ascending=False).head(30)


In [ ]:
# Aggregate weighted MSE by signal/grouping after applying group-specific best alpha

def aggregate_group_alpha_result(data, signal_col, group_cols=None, alpha_grid=None, min_n=25):
    summary, d = eval_alpha_grid(data, signal_col, group_cols=group_cols, alpha_grid=alpha_grid, min_n=min_n)
    if summary.empty:
        return None
    if group_cols is None:
        group_cols_use = ['_group']
        d['_group'] = 'all'
    else:
        group_cols_use = group_cols
    # map alpha back
    key_to_alpha = {}
    for _, row in summary.iterrows():
        key = tuple(row[c] for c in group_cols_use)
        key_to_alpha[key] = row['best_alpha']
    preds = []
    for _, row in d.iterrows():
        key = tuple(row[c] for c in group_cols_use)
        a = key_to_alpha.get(key, 0.0)
        preds.append(row['cross_pred'] + a * row[signal_col])
    preds = np.array(preds, float)
    base_mse = np.mean((d['cross_pred'].to_numpy(float) - d['true_iv'].to_numpy(float))**2)
    new_mse = np.mean((preds - d['true_iv'].to_numpy(float))**2)
    return {
        'signal': signal_col,
        'grouping': 'global' if group_cols is None else '+'.join(group_cols),
        'n': len(d),
        'base_mse': base_mse,
        'new_mse': new_mse,
        'improvement_pct': (base_mse - new_mse) / base_mse * 100 if base_mse > 0 else np.nan,
        'nonzero_alpha_groups': int((summary['best_alpha'].abs() > 1e-12).sum()),
        'groups': len(summary),
    }

agg_results = []
for sig in signals:
    for group_cols in [None, ['option_type'], ['option_type','k_rank'], ['is_jan27','option_type'], ['is_jan27','option_type','k_rank']]:
        r = aggregate_group_alpha_result(resid_df, sig, group_cols=group_cols, min_n=25)
        if r:
            agg_results.append(r)
agg_results = pd.DataFrame(agg_results).sort_values('improvement_pct', ascending=False)
agg_results


In [ ]:
# Visualize best correction candidates
fig, ax = plt.subplots(figsize=(14,5))
top = agg_results.head(20).copy()
top['label'] = top['signal'] + ' | ' + top['grouping']
ax.barh(top['label'][::-1], top['improvement_pct'][::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('CV Improvement from Underlying-derived Residual Corrections')
ax.set_xlabel('Improvement vs cross-section baseline (%)')
plt.show()


## 10. Large spot move regimes

Maybe spot signal only matters after big moves. This section evaluates residual error and correction performance conditional on absolute spot move buckets.


In [ ]:
# ================================================================
# 14. Big move regime analysis
# ================================================================

r = resid_df.copy()
r['abs_spot_bucket'] = pd.cut(
    r['abs_spot_ret_1_bp'],
    bins=[-np.inf, 2, 5, 10, 20, np.inf],
    labels=['<=2bp','2-5bp','5-10bp','10-20bp','>20bp']
)

bucket_resid = r.groupby(['is_jan27','option_type','abs_spot_bucket']).agg(
    n=('sq_err','size'),
    base_mse=('sq_err','mean'),
    mean_abs_residual=('residual', lambda x: np.mean(np.abs(x))),
    mean_spot_abs_bp=('abs_spot_ret_1_bp','mean'),
).reset_index()
bucket_resid


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,5), sharey=True)
for ax, ot in zip(axes, ['CE','PE']):
    sub = bucket_resid[bucket_resid['option_type']==ot]
    for isj in [False, True]:
        ss = sub[sub['is_jan27']==isj]
        ax.plot(ss['abs_spot_bucket'].astype(str), ss['base_mse'], marker='o', label=f'Jan27={isj}')
    ax.set_title(f'{ot}: residual MSE by abs spot move bucket')
    ax.set_xlabel('|spot return| bucket')
    ax.set_ylabel('MSE')
    ax.legend()
plt.xticks(rotation=30)
plt.show()


In [ ]:
# Evaluate corrections only on big-move buckets
big = resid_df[resid_df['abs_spot_ret_1_bp'] >= 10].copy()
print('Big move residual rows:', len(big), 'base MSE:', big['sq_err'].mean() if len(big) else np.nan)
if len(big):
    big_results = []
    for sig in signals:
        for group_cols in [None, ['option_type'], ['option_type','k_rank'], ['is_jan27','option_type','k_rank']]:
            rr = aggregate_group_alpha_result(big, sig, group_cols=group_cols, min_n=10)
            if rr:
                big_results.append(rr)
    display(pd.DataFrame(big_results).sort_values('improvement_pct', ascending=False))


## 11. Option-specific temporal relationship

Some options may react to spot moves more strongly than others. This section builds per-contract summaries.


In [ ]:
# ================================================================
# 15. Per-contract signal strength
# ================================================================

contract_rows = []
for c, g in long_move2.groupby('contract'):
    for feat in ['spot_ret_1_bp','abs_spot_ret_1_bp','slope_x_dmoneyness','spot_realized_abs_5_bp']:
        res = corr_safe(g[feat], g['iv_change_1'])
        contract_rows.append({
            'contract': c,
            'option_type': type_map[c],
            'strike': strike_map[c],
            'k_rank': rank_map[c],
            'feature': feat,
            **res.to_dict(),
        })
contract_corr = pd.DataFrame(contract_rows)
contract_corr['abs_pearson'] = contract_corr['pearson'].abs()
contract_corr.sort_values('abs_pearson', ascending=False).head(30)


In [ ]:
# Plot per-contract correlations by rank for key features
for feat in ['spot_ret_1_bp','abs_spot_ret_1_bp','slope_x_dmoneyness']:
    sub = contract_corr[contract_corr['feature']==feat]
    fig, ax = plt.subplots(figsize=(13,4))
    for ot in ['CE','PE']:
        ss = sub[sub['option_type']==ot]
        ax.plot(ss['k_rank'], ss['pearson'], marker='o', label=ot)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'Per-contract correlation: {feat} vs IV change')
    ax.set_xlabel('k_rank')
    ax.set_ylabel('Pearson corr')
    ax.legend()
    plt.show()


## 12. Jan 27 deep dive

The project context suggests Jan 27 is special. This section isolates Jan 27 and compares spot movement, IV movement, cross-section residuals, and temporal behavior.


In [ ]:
# ================================================================
# 16. Jan27 specific panels
# ================================================================

jdf = df[df['is_jan27']].copy()
print('Jan27 rows:', len(jdf))

if len(jdf):
    fig, ax1 = plt.subplots(figsize=(14,5))
    ax1.plot(jdf['datetime_parsed'], jdf['underlying_price'], label='Underlying', linewidth=1.6)
    ax1.set_ylabel('Underlying')
    ax2 = ax1.twinx()
    # median IV by timestamp
    med_iv = jdf[option_cols].median(axis=1, skipna=True)
    ax2.plot(jdf['datetime_parsed'], med_iv, label='Median IV', linestyle='--', linewidth=1.4)
    ax2.set_ylabel('Median IV')
    ax1.set_title('Jan27: Underlying Price and Median IV')
    ax1.tick_params(axis='x', rotation=30)
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1+lines2, labels1+labels2, loc='best')
    plt.show()

    fig, ax = plt.subplots(figsize=(14,4))
    ax.plot(jdf['datetime_parsed'], jdf['spot_ret_1_bp'], marker='o', linewidth=1)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title('Jan27: Spot Return bp')
    ax.set_ylabel('bp')
    plt.xticks(rotation=30)
    plt.show()


In [ ]:
# Jan27 IV change heatmap by contract rank/time
jmove = long_move2[long_move2['is_jan27']].copy()
if len(jmove):
    for ot in ['CE','PE']:
        sub = jmove[jmove['option_type']==ot]
        piv = sub.pivot_table(index='datetime_parsed', columns='k_rank', values='iv_change_1', aggfunc='mean')
        fig, ax = plt.subplots(figsize=(14,6))
        if HAS_SEABORN:
            sns.heatmap(piv, center=0, cmap='coolwarm', ax=ax)
        else:
            im = ax.imshow(piv.values, aspect='auto')
            fig.colorbar(im, ax=ax)
        ax.set_title(f'Jan27 {ot}: IV Change Heatmap by Time × Strike Rank')
        ax.set_xlabel('k_rank')
        ax.set_ylabel('timestamp')
        plt.show()
else:
    print('No Jan27 movement rows.')


In [ ]:
# Jan27 residual analysis
jres = resid_df[resid_df['is_jan27']].copy()
print('Jan27 residual rows:', len(jres), 'MSE:', jres['sq_err'].mean() if len(jres) else np.nan)
if len(jres):
    display(jres.groupby(['option_type','k_rank']).agg(
        n=('sq_err','size'),
        mse=('sq_err','mean'),
        mean_resid=('residual','mean'),
        mean_abs_resid=('residual', lambda x: np.mean(np.abs(x))),
        mean_spot_bp=('spot_ret_1_bp','mean'),
    ).reset_index().sort_values('mse', ascending=False).head(20))

    for sig in ['spot_ret_1_bp','abs_spot_ret_1_bp','slope_x_dmoneyness','spot_realized_abs_5_bp']:
        summ, _ = eval_alpha_grid(jres, sig, group_cols=['option_type','k_rank'], min_n=8,
                                  alpha_grid=np.round(np.arange(-2,2.001,0.05),3))
        print('\nSignal:', sig)
        display(summ.sort_values('improvement_pct', ascending=False).head(10))


## 13. A practical candidate feature: slope × moneyness drift

This is the most theoretically aligned spot feature.

When spot changes, fixed strike moves horizontally in moneyness:

\[
x_{t,c} = K_c / S_t
\]

\[
\Delta x_{t,c} = K_c/S_t - K_c/S_{t-1}
\]

If the local smile slope is \(\partial IV / \partial x\), then the IV movement induced by the horizontal shift is roughly:

\[
\Delta IV \approx \frac{\partial IV}{\partial x}\Delta x
\]

So we test whether:

\[
signal = local\_smile\_slope \times \Delta moneyness
\]

explains either actual IV movement or cross-section residuals.


In [ ]:
# ================================================================
# 17. Focused slope × dmoneyness diagnostics
# ================================================================

focus = resid_df[np.isfinite(resid_df['slope_x_dmoneyness'])].copy()
print('Rows with signal:', len(focus), 'base MSE:', focus['sq_err'].mean())

# Grid correction globally and by option/rank
for grouping in [None, ['option_type'], ['option_type','k_rank'], ['is_jan27','option_type','k_rank']]:
    rr = aggregate_group_alpha_result(focus, 'slope_x_dmoneyness', group_cols=grouping, min_n=20)
    print(rr)


In [ ]:
# Distribution of suggested correction magnitude if alpha=1
fig, ax = plt.subplots(figsize=(12,4))
ax.hist(focus['slope_x_dmoneyness'].dropna(), bins=100)
ax.set_title('Distribution of slope × dmoneyness signal')
ax.set_xlabel('signal magnitude in IV units')
ax.set_ylabel('Count')
plt.show()

# Compare residual and signal by rank
rank_sig = focus.groupby(['is_jan27','option_type','k_rank']).agg(
    n=('residual','size'),
    mean_resid=('residual','mean'),
    mean_signal=('slope_x_dmoneyness','mean'),
    corr=('residual', lambda y: np.nan),
).reset_index()

# add corr manually
corrs = []
for keys, g in focus.groupby(['is_jan27','option_type','k_rank']):
    corrs.append((*keys, corr_safe(g['slope_x_dmoneyness'], g['residual'])['pearson']))
corr_df = pd.DataFrame(corrs, columns=['is_jan27','option_type','k_rank','corr_signal_resid'])
rank_sig = rank_sig.drop(columns=['corr']).merge(corr_df, on=['is_jan27','option_type','k_rank'], how='left')
rank_sig.sort_values('corr_signal_resid', key=lambda s: s.abs(), ascending=False).head(30)


## 14. What to put into the imputer if validation passes

The cells below generate a compact report of which signals actually improved CV. Use this as the decision gate.

Recommended policy:

- Only add signals with positive aggregate CV improvement.
- Avoid many tiny per-rank coefficients unless the improvement is clearly stable.
- Prefer interpretable features: `slope_x_dmoneyness`, `abs_spot_ret_1_bp`, or `spot_realized_abs_5_bp`.
- Use `alpha = 0` default whenever a bucket has low validation count.


In [ ]:
# ================================================================
# 18. Final EDA report tables
# ================================================================

# Overall baseline error by regime
report_baseline = resid_df.groupby(['is_jan27','option_type']).agg(
    n=('sq_err','size'),
    mse=('sq_err','mean'),
    rmse=('sq_err', lambda x: np.sqrt(np.mean(x))),
    mean_abs_resid=('residual', lambda x: np.mean(np.abs(x))),
).reset_index()

# Candidate correction summary
candidate_report = agg_results.copy()

print('Baseline residual error by regime:')
display(report_baseline)

print('Candidate underlying-price correction results:')
display(candidate_report.sort_values('improvement_pct', ascending=False).head(20))

# Save outputs
OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
report_baseline.to_csv(OUT_DIR / 'underlying_iv_eda_baseline_residual_by_regime.csv', index=False)
candidate_report.to_csv(OUT_DIR / 'underlying_iv_eda_candidate_corrections.csv', index=False)
resid_corr_regime.to_csv(OUT_DIR / 'underlying_iv_eda_residual_correlations_by_regime.csv', index=False)
contract_corr.to_csv(OUT_DIR / 'underlying_iv_eda_contract_correlations.csv', index=False)
print('Saved EDA CSV reports to:', OUT_DIR)


In [ ]:
# ================================================================
# 19. Human-readable conclusion generator
# ================================================================

best = candidate_report.sort_values('improvement_pct', ascending=False).iloc[0]
base_mse = float(resid_df['sq_err'].mean())
print('\n' + '='*80)
print('UNDERLYING PRICE SIGNAL EDA — SUMMARY')
print('='*80)
print(f'Interior masked-residual baseline MSE: {base_mse:.9f}')
print('\nBest tested correction:')
print(f"  signal   : {best['signal']}")
print(f"  grouping : {best['grouping']}")
print(f"  new MSE  : {best['new_mse']:.9f}")
print(f"  improve  : {best['improvement_pct']:.3f}%")
print(f"  groups   : {int(best['groups'])}, nonzero alpha groups: {int(best['nonzero_alpha_groups'])}")

if best['improvement_pct'] > 0.5:
    print('\nRecommendation: This signal is worth implementing behind a CV gate.')
elif best['improvement_pct'] > 0:
    print('\nRecommendation: Tiny positive signal. Implement only if public/private validation agrees.')
else:
    print('\nRecommendation: Do not add underlying-price correction yet; current CV does not support it.')

print('='*80)


# Notes for converting EDA into code

If the best signal is `slope_x_dmoneyness`, the implementation idea is:

1. For a missing interior cell, compute your current cross-section prediction.
2. Compute local smile slope from observed same-row same-option points.
3. Compute `dmoneyness = K/S_t - K/S_{t-1}`.
4. Compute `signal = local_slope * dmoneyness`.
5. Add `alpha * signal`, where alpha was chosen by CV.
6. Clip the correction gently, for example to `±0.005` or `±0.01`, and validate again.

Do **not** add this blindly. The final table above tells you whether the signal helps on your masked validation.
